# 三種不同的模型的資料前處理

## 共同部分
### 切割資料集
最先執行，並且固定隨機變數以利重現

### 刪除冗餘變數
刪除 risk_score, credit_score, customer_segment, user_id 變數

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import yaml
from pathlib import Path

# 載入 config.yaml
PROJECT_ROOT = Path("..")
with open(PROJECT_ROOT / "config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 1. 載入 interim 資料
interim_path = PROJECT_ROOT / config['data']['interim_dir'] / "01_data_understanding.csv"
print(f"📖 載入中間資料: {interim_path}")
df = pd.read_csv(interim_path)
print(f"✅ 資料已載入，形狀: {df.shape}")

# 2. 切割資料集 (80% 訓練, 20% 測試，以 default_flag 進行分層抽樣)
X = df.drop(columns=['default_flag'])
y = df['default_flag']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"分割完成 | 訓練集形狀: {X_train.shape}, 測試集形狀: {X_test.shape}")

# 3. 刪除冗餘變數: risk_score, credit_score, customer_segment, user_id
redundant_cols = ['risk_score', 'credit_score', 'customer_segment', 'user_id']
X_train = X_train.drop(columns=[col for col in redundant_cols if col in X_train.columns])
X_test = X_test.drop(columns=[col for col in redundant_cols if col in X_test.columns])
print(f"已刪除冗餘變數，特徵數量: {X_train.shape[1]}")

📖 載入中間資料: ..\data\interim\01_data_understanding.csv
✅ 資料已載入，形狀: (10345, 21)
分割完成 | 訓練集形狀: (8276, 20), 測試集形狀: (2069, 20)
已刪除冗餘變數，特徵數量: 16


## 1. 隨機森林 (Random Forest)

隨機森林是由多棵決策樹透過 Bagging 組成的集成模型。它的特性是「基於規則切割」，不計算空間距離。

實作
- 類別變數編碼 One-Hot Encoding

In [2]:
# 1. One-Hot 編碼類別型變數 (employment_type, product_category, location)
categorical_cols = ['employment_type', 'product_category', 'location']

# 分別編碼訓練集與測試集
X_train_rf = pd.get_dummies(X_train, columns=categorical_cols, dtype=int)
X_test_rf = pd.get_dummies(X_test, columns=categorical_cols, dtype=int)

# 對齊測試集的欄位，避免類別不一致導致特徵數量不對等，缺失特徵以 0 填充
X_test_rf = X_test_rf.reindex(columns=X_train_rf.columns, fill_value=0)

# 合併 target 變數
train_rf = pd.concat([X_train_rf, y_train], axis=1)
test_rf = pd.concat([X_test_rf, y_test], axis=1)

# 2. 儲存處理完成的資料
processed_dir = PROJECT_ROOT / config['data']['processed_dir']
processed_dir.mkdir(parents=True, exist_ok=True)

train_rf.to_csv(processed_dir / "rf_train.csv", index=False)
test_rf.to_csv(processed_dir / "rf_test.csv", index=False)
print(f"✅ 隨機森林前處理完成！資料已儲存到 {processed_dir}")
print(f"訓練集大小: {train_rf.shape}, 測試集大小: {test_rf.shape}")

✅ 隨機森林前處理完成！資料已儲存到 ..\data\processed
訓練集大小: (8276, 29), 測試集大小: (2069, 29)


## 2. 梯度提升樹 (XGBoost / LightGBM)

這是目前在 Kaggle 表格型資料中最常勝的演算法。由多棵決策樹透過 Boosting（迭代修正錯誤）組成。

實作
- 類別變數編碼 One-Hot Encoding

In [3]:
# 由於 XGBoost/LightGBM 前處理要求與隨機森林一致 (無需 scaling，保留 outliers，進行 One-Hot 編碼)
# 我們直接複用 RF 處理完的特徵結構，並儲存為獨立的 xgb 檔案，便於後續模型訓練單獨載入
train_rf.to_csv(processed_dir / "xgb_train.csv", index=False)
test_rf.to_csv(processed_dir / "xgb_test.csv", index=False)
print(f"✅ 梯度提升樹前處理完成！資料已儲存到 {processed_dir}")
print(f"訓練集大小: {train_rf.shape}, 測試集大小: {test_rf.shape}")

✅ 梯度提升樹前處理完成！資料已儲存到 ..\data\processed
訓練集大小: (8276, 29), 測試集大小: (2069, 29)


## 3. 深度神經網路 (Deep Neural Networks / MLP)

神經網路透過反向傳播 (Backpropagation) 與梯度下降法 (Gradient Descent) 更新權重。它對資料的數值分佈與尺度極度敏感。

實作
1. 離群值處理:  Winsorization (截斷)將五種有離群值變數找出 Training set 的 99% 分位數，將所有大於該分位數的值，強制覆蓋為 99% 分位數的值。
2. 類別變數編碼 One-Hot Encoding
3. 特徵縮放 StandardScaler: 將所有數值特徵轉為平均值 0、標準差 1

In [4]:
# 1. 複製特徵資料副本
X_train_dnn = X_train.copy()
X_test_dnn = X_test.copy()

# 2. 離群值處理: Winsorization (截斷)
# 針對五種有離群值的數值變數: monthly_income, repayment_delay_days, debt_to_income_ratio, purchase_amount, missed_payments
outlier_cols = ['monthly_income', 'repayment_delay_days', 'debt_to_income_ratio', 'purchase_amount', 'missed_payments']

for col in outlier_cols:
    if col in X_train_dnn.columns:
        # 僅使用 Training set 的 99% 分位數作為閾值 (避免 data leakage)
        percentile_99 = X_train_dnn[col].quantile(0.99)
        # 將大於該分位數的值，強制覆蓋為該分位數的值
        X_train_dnn[col] = np.where(X_train_dnn[col] > percentile_99, percentile_99, X_train_dnn[col])
        X_test_dnn[col] = np.where(X_test_dnn[col] > percentile_99, percentile_99, X_test_dnn[col])
        print(f"Winsorized {col} 於 99% 分位數: {percentile_99}")

# 3. One-Hot 編碼類別變數
X_train_dnn = pd.get_dummies(X_train_dnn, columns=categorical_cols, dtype=int)
X_test_dnn = pd.get_dummies(X_test_dnn, columns=categorical_cols, dtype=int)
X_test_dnn = X_test_dnn.reindex(columns=X_train_dnn.columns, fill_value=0)

# 4. 特徵縮放: StandardScaler (平均值 0, 標準差 1)
# 僅對數值型特徵進行標準化 (排除 One-Hot 欄位)
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

scaler = StandardScaler()
# 僅 fit 於 Training set，再 transform 到 Training set 與 Test set (避免 data leakage)
X_train_dnn[numerical_cols] = scaler.fit_transform(X_train_dnn[numerical_cols])
X_test_dnn[numerical_cols] = scaler.transform(X_test_dnn[numerical_cols])
print(f"✅ 已對數值欄位進行 StandardScaler 標準化: {numerical_cols}")

# 合併 target 變數
train_dnn = pd.concat([X_train_dnn, y_train], axis=1)
test_dnn = pd.concat([X_test_dnn, y_test], axis=1)

# 5. 儲存 DNN 專用資料集
train_dnn.to_csv(processed_dir / "dnn_train.csv", index=False)
test_dnn.to_csv(processed_dir / "dnn_test.csv", index=False)
print(f"✅ 深度神經網路前處理完成！資料已儲存到 {processed_dir}")
print(f"訓練集大小: {train_dnn.shape}, 測試集大小: {test_dnn.shape}")

Winsorized monthly_income 於 99% 分位數: 104730.29000000001
Winsorized repayment_delay_days 於 99% 分位數: 24.0
Winsorized debt_to_income_ratio 於 99% 分位數: 0.5220708180466775
Winsorized purchase_amount 於 99% 分位數: 5000.0
Winsorized missed_payments 於 99% 分位數: 4.0
✅ 已對數值欄位進行 StandardScaler 標準化: ['age', 'monthly_income', 'purchase_amount', 'bnpl_installments', 'repayment_delay_days', 'missed_payments', 'app_usage_frequency', 'debt_to_income_ratio', 'transaction_year', 'transaction_month', 'transaction_day', 'transaction_dayofweek', 'transaction_is_weekend']
✅ 深度神經網路前處理完成！資料已儲存到 ..\data\processed
訓練集大小: (8276, 29), 測試集大小: (2069, 29)
